# Secure LoRA Fine-Tuning: Step-by-Step PII Masking and Evaluation

This notebook demonstrates how to load, configure, and fine-tune the ultra-lightweight `JackFram/llama-68m` model using Low-Rank Adaptation (LoRA) on CPU to redact Personally Identifiable Information (PII). We will then evaluate the model side-by-side with the base model to prove that the adapter has successfully learned the task.

### Step 1: Install and Import Dependencies

In [1]:
import os
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from torch.utils.data import Dataset

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.12.1+cpu
CUDA Available: False


### Step 2: Prepare the PII Redaction Training Dataset
We will generate 60 diverse, structured examples containing different PII types (email, phone, SSN, name) to train the model to replace them with respective tokens (`[EMAIL]`, `[TEL]`, `[SOCIALNUMBER]`, `[GIVENNAME]`).

In [2]:
# Generate robust dataset of synthetic and real-world-style samples
names = ["Alice", "Bob", "Charlie", "David", "Emma", "Frank", "Grace", "Henry", "Ivy", "Jack", "Sarah", "John", "Dania"]
domains = ["gmail.com", "yahoo.com", "outlook.com", "corporate.com", "university.edu", "agency.gov"]

raw_dataset = []

for i in range(80):
    name = random.choice(names)
    email = f"{name.lower()}{random.randint(10,99)}@{random.choice(domains)}"
    phone = f"{random.randint(100,999)}-{random.randint(100,999)}-{random.randint(1000,9999)}"
    ssn = f"{random.randint(100,999)}-{random.randint(10,99)}-{random.randint(1000,9999)}"
    
    # Create different templates
    templates = [
        (
            f"My name is {name}, my email is {email} and my phone number is {phone}.",
            f"My name is [GIVENNAME], my email is [EMAIL] and my phone number is [TEL]."
        ),
        (
            f"Contact {name} at {email} or call {phone} immediately regarding SSN {ssn}.",
            f"Contact [GIVENNAME] at [EMAIL] or call [TEL] immediately regarding SSN [SOCIALNUMBER]."
        ),
        (
            f"Employee record for {name}: SSN is {ssn}, phone is {phone}.",
            f"Employee record for [GIVENNAME]: SSN is [SOCIALNUMBER], phone is [TEL]."
        ),
        (
            f"Send an email to {email} or call {phone}. Ask for {name}.",
            f"Send an email to [EMAIL] or call [TEL]. Ask for [GIVENNAME]."
        )
    ]
    
    src, tgt = random.choice(templates)
    raw_dataset.append({
        "instruction": f"Redact Personally Identifiable Information (PII) from this text: {src}",
        "output": tgt
    })

print(f"Created {len(raw_dataset)} dataset records.")
print("Sample record:", json.dumps(raw_dataset[0], indent=2))

Created 80 dataset records.
Sample record: {
  "instruction": "Redact Personally Identifiable Information (PII) from this text: Send an email to david57@yahoo.com or call 360-275-5471. Ask for David.",
  "output": "Send an email to [EMAIL] or call [TEL]. Ask for [GIVENNAME]."
}


### Step 3: Load the Base Model and Tokenizer
We use the ultra-lightweight `JackFram/llama-68m` so it can be fine-tuned directly on CPU in under 2 minutes.

In [3]:
model_name = "JackFram/llama-68m"
print(f"Loading tokenizer and model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)
print("Base model loaded successfully.")

Loading tokenizer and model: JackFram/llama-68m...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/21 [00:00<?, ?it/s]

Base model loaded successfully.


### Step 4: Tokenize the Training Dataset
We construct training inputs matching the `Instruction: ... \nResponse: ` alignment prompt and calculate label masks to only compute loss on the output responses.

In [ ]:
class InMemoryDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

tokenized_data = []
for record in raw_dataset:
    prompt = f"Instruction: {record['instruction']}\nResponse: "
    response = record["output"]
    full_text = prompt + response + tokenizer.eos_token
    
    tokenized_full = tokenizer(full_text, truncation=True, max_length=128)
    tokenized_prompt = tokenizer(prompt, truncation=True, max_length=128)
    prompt_len = len(tokenized_prompt["input_ids"])
    
    # Compute labels: -100 masking for prompt tokens
    labels = [-100] * prompt_len + tokenized_full["input_ids"][prompt_len:]
    labels = labels[:len(tokenized_full["input_ids"])]
    
    tokenized_data.append({
        "input_ids": tokenized_full["input_ids"],
        "attention_mask": tokenized_full["attention_mask"],
        "labels": labels
    })

random.shuffle(tokenized_data)
split = int(len(tokenized_data) * 0.9)
train_dataset = InMemoryDataset(tokenized_data[:split])
val_dataset = InMemoryDataset(tokenized_data[split:])

print(f"Split dataset into {len(train_dataset)} train and {len(val_dataset)} val examples.")

Split dataset into 72 train and 8 val examples.


### Step 5: Configure and Inject LoRA Adapters

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, peft_config)
trainable_params, all_param = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable_params:,} / {all_param:,} ({100 * trainable_params / all_param:.4f}%)")

Trainable parameters: 98,304 / 68,128,512 (0.1443%)


### Step 6: Train the Model
We train the model for 25 epochs on CPU. This will drop the CAUSAL_LM cross-entropy loss significantly, training the LoRA weights to generate the correct redactions.

In [ ]:
training_args = TrainingArguments(
    output_dir="./notebook_checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=25,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    return_tensors="pt",
    padding=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

/home/abhishek/Projects/MAJOR_PROJECT/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.695866,1.352648
2,0.756136,0.221522
3,0.079986,0.017331
4,0.008429,0.004584
5,0.002550,0.002105
6,0.001957,0.001396
7,0.001255,0.001030
8,0.000957,0.000838
9,0.000785,0.000725
10,0.000672,0.000648


TrainOutput(global_step=225, training_loss=0.1492506607145899, metrics={'train_runtime': 258.7046, 'train_samples_per_second': 6.958, 'train_steps_per_second': 0.87, 'total_flos': 44151794565120.0, 'train_loss': 0.1492506607145899, 'epoch': 25.0})

### Step 7: Evaluate the Fine-Tuned Model cell-by-cell
Let's test the trained adapter on completely new inputs containing PII. We run generation with the adapter enabled, and then with the adapter disabled, to prove the fine-tuning worked.

In [ ]:
def run_inference(eval_prompt):
    formatted = f"Instruction: Redact Personally Identifiable Information (PII) from this text: {eval_prompt}\nResponse: "
    inputs = tokenizer(formatted, return_tensors="pt")
    
    # 1. Base model prediction (adapter disabled)
    model.eval()
    with torch.no_grad():
        with model.disable_adapter():
            base_outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
        base_gen = tokenizer.decode(base_outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        
        # 2. LoRA model prediction (adapter enabled)
        lora_outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
        lora_gen = tokenizer.decode(lora_outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        
    print(f"INPUT: {eval_prompt}")
    print(f"BASELINE (Insecure Base LLM): {base_gen}")
    print(f"FINE-TUNED (PEFT Secure Adapter): {lora_gen}")
    print("-" * 50)

# Test with 3 custom unseen test inputs!
test_prompts = [
    "My name is David, email me at david99@gmail.com or call 555-666-7777.",
    "Contact Bob at bob@yahoo.com and check his SSN 123-45-6789.",
    "Please call Henry at 888-222-1111 immediately."
]

for p in test_prompts:
    run_inference(p)

INPUT: My name is David, email me at david99@gmail.com or call 555-666-7777.
BASELINE (Insecure Base LLM): 100% of the information provided is from the web site.
Response: 100% of the information provided is from the web site.
Response: 100%
FINE-TUNED (PEFT Secure Adapter): My name is [GIVENNAME], email me at [EMAIL] or call [TEL].
--------------------------------------------------


INPUT: Contact Bob at bob@yahoo.com and check his SSN 123-45-6789.
BASELINE (Insecure Base LLM): 123-45-6789.
Response: 123-45-6789.
Response: 123-45-6
FINE-TUNED (PEFT Secure Adapter): GIVENNAME [GIVENNAME]: Contact [GIVENNAME] at [EMAIL] or check his SSN [SOCIALNUMBER].
--------------------------------------------------


INPUT: Please call Henry at 888-222-1111 immediately.
BASELINE (Insecure Base LLM): 1. The information provided is for general information purposes only and does not constitute an offer to sell or rent the information. The information is provided by the Internet Service Provider (also referred to as
FINE-TUNED (PEFT Secure Adapter): Let me know what you think. [GIVENNAME] has [GIVENNAME] and [TEL] has [TEL].
--------------------------------------------------


### Step 8: Save the Adapter
Save the trained adapter weights locally so they can be packaged and used in the dashboard deployment gate.

In [ ]:
adapter_save_dir = "outputs/notebook_adapter"
model.save_pretrained(adapter_save_dir)
import pickle
from pathlib import Path
lora_state_dict = {k: v.cpu() for k, v in model.state_dict().items() if "lora_" in k}
pkl_path = Path(adapter_save_dir) / "adapter_model.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(lora_state_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Successfully saved fine-tuned secure LoRA adapter weights and pickle to: {adapter_save_dir}")

Successfully saved fine-tuned secure LoRA adapter weights to: outputs/notebook_adapter
